# TetraFT — Build FineWeb-Edu sample (Kaggle)

**Goal:** Stream `HuggingFaceFW/fineweb-edu` → fixed `train.jsonl` + held-out `val.jsonl`.

| Setting | Value |
|---------|--------|
| Accelerator | **None** (CPU) |
| Internet | **ON** |
| **SAMPLE** | `"400m"` (marathon) or `"50m"` (legacy) |
| Seed | 42 |

| SAMPLE | Train tok | Val tok | Dataset name | Wall (est.) |
|--------|----------:|--------:|--------------|-------------|
| `50m` | 50M | 0.5M | `tetraft-fineweb-edu-50m` | ~1–3 h |
| **`400m`** | **400M** | **0.5M** | **`tetraft-fineweb-edu-400m`** | **~8–20 h** |

**Prereq:** Attach Dataset `tetraft-code` (flat repo `.py` files).

**After success:** Save Version → include output → create Dataset from `/kaggle/working/fineweb-*/`.

In [ ]:
%pip install -q datasets huggingface_hub

In [ ]:
import os
import sys
import logging
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)

CANDIDATES = [
    Path("/kaggle/input/tetraft-code"),
    Path("/kaggle/working"),
    Path("."),
]
inp = Path("/kaggle/input")
if inp.is_dir():
    for sub in sorted(inp.iterdir()):
        if sub.is_dir():
            CANDIDATES.append(sub)

code_root = None
for root in CANDIDATES:
    if (root / "data.py").is_file():
        code_root = root
        break

if code_root is None:
    raise FileNotFoundError(
        "data.py not found. Attach Kaggle Dataset 'tetraft-code' with flat .py modules."
    )

sys.path.insert(0, str(code_root))
print("Using code from:", code_root)

In [ ]:
from data import build_fineweb_sample
from pathlib import Path

# =============================================================================
# SAMPLE: "400m" for heal_kl_trust_400m marathon | "50m" legacy scouts
# =============================================================================
SAMPLE = "400m"  # <-- 400m | 50m

if SAMPLE == "400m":
    OUT = Path("/kaggle/working/fineweb-400m")
    MAX_TRAIN = 400_000_000
    MAX_VAL = 500_000
    DATASET_NAME = "tetraft-fineweb-edu-400m"
elif SAMPLE == "50m":
    OUT = Path("/kaggle/working/fineweb-50m")
    MAX_TRAIN = 50_000_000
    MAX_VAL = 500_000
    DATASET_NAME = "tetraft-fineweb-edu-50m"
else:
    raise ValueError("SAMPLE must be '400m' or '50m'")

OUT.mkdir(parents=True, exist_ok=True)
print(f"SAMPLE={SAMPLE} → {OUT} train={MAX_TRAIN:,} val={MAX_VAL:,}")
print(f"Publish as Kaggle Dataset: {DATASET_NAME}")

meta = build_fineweb_sample(
    output_dir=OUT,
    max_train_tokens=MAX_TRAIN,
    max_val_tokens=MAX_VAL,
    seed=42,
)
print(meta)

In [ ]:
import json
from pathlib import Path

OUT = Path("/kaggle/working/fineweb-400m")
if not (OUT / "train.jsonl").is_file():
    OUT = Path("/kaggle/working/fineweb-50m")

for name in ("train.jsonl", "val.jsonl", "sample_meta.json"):
    p = OUT / name
    assert p.is_file(), f"missing {p}"
    print(f"{name}: {p.stat().st_size / 1e6:.1f} MB")

with open(OUT / "sample_meta.json") as f:
    meta = json.load(f)
print(json.dumps(meta, indent=2))

with open(OUT / "val.jsonl") as f:
    row = json.loads(f.readline())
print("val sample keys:", row.keys())
print("val text preview:", row["text"][:200].replace("\n", " "))

## Save as Kaggle Dataset

1. **Save Version** (Save & Run All).
2. Enable **output** in the save dialog.
3. Open the completed version → **New Dataset** from output.
4. Name: `tetraft-fineweb-edu-400m` (or `…-50m`).
5. Keep files under `fineweb-400m/` (or `fineweb-50m/`):
   - `train.jsonl`
   - `val.jsonl`
   - `sample_meta.json`

Next: attach `tetraft-code` + FineWeb dataset and run `notebooks/run_smoke.ipynb` with `SESSION=1`.